# Entrenamiento del modelo: gato vs. perro

**Proyecto Medio de Visión Artificial** · basado en el ejemplo de Keras
[Image classification from scratch](https://keras.io/examples/vision/image_classification_from_scratch/),
siguiendo los pasos vistos en clase:

1. Descargar el dataset
2. **Limpiar los datos** (quitar imágenes dañadas)
3. **Balancear las clases** (misma cantidad de gatos y perros)
4. **Partir el dataset**: entrenamiento 70 %, validación 15 % y prueba 15 % (`train.csv`, `val.csv`, `test.csv`)
5. Cargar las imágenes desde los CSV con `tf.data`
6. Construir el modelo desde cero
7. Entrenar con *callbacks*
8. **Evaluar con el conjunto de prueba**: accuracy, precision, recall, specificity y matriz de confusión
9. Guardar `modelo.keras` para el servidor Flask

**Cómo ejecutarlo en Google Colab:** sube este archivo a [colab.research.google.com](https://colab.research.google.com),
ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU* y luego
*Entorno de ejecución → Ejecutar todo*. Con GPU tarda alrededor de una hora; sin GPU, muchas horas.

## 0. Configuración

In [ ]:
import glob
import os
import random
import urllib.request
import zipfile

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

SEMILLA = 1337                 # para que el resultado se pueda repetir
IMAGE_SIZE = (180, 180)        # el servidor Flask usa este mismo tamaño
BATCH_SIZE = 128
EPOCHS = 25
MUESTRA_POR_CLASE = None       # None = todas las imágenes. Un número (ej. 300) = prueba rápida
CARPETA_DATOS = "PetImages"

keras.utils.set_random_seed(SEMILLA)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NO HAY GPU. En Colab: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU")

## 1. Descargar el dataset

Cats vs. Dogs de Kaggle/Microsoft (786 MB). Queda una carpeta `PetImages` con dos subcarpetas: `Cat` y `Dog`.

In [ ]:
URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
ZIP = "kagglecatsanddogs_5340.zip"

if not os.path.isdir(CARPETA_DATOS):
    if not os.path.exists(ZIP):
        print("Descargando el dataset (786 MB)...")
        urllib.request.urlretrieve(URL, ZIP)
    print("Descomprimiendo...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(".")

for clase in ("Cat", "Dog"):
    print(clase, "->", len(os.listdir(os.path.join(CARPETA_DATOS, clase))), "archivos")

## 2. Limpieza de los datos

Los datasets reales traen archivos dañados. Se hacen dos revisiones y se borra lo que no sirve:

1. **Cabecera JFIF**: un JPG bien codificado tiene el texto `JFIF` al inicio del archivo (es el filtro del ejemplo de Keras).
   Aquí también caen los archivos vacíos y los que no son imágenes (`Thumbs.db`).
2. **Que se pueda abrir**: se intenta decodificar cada imagen con TensorFlow; si falla, se elimina.

Esta celda tarda un par de minutos porque abre las 25 000 imágenes.

In [ ]:
eliminadas = {"Cat": 0, "Dog": 0}

for clase in ("Cat", "Dog"):
    carpeta = os.path.join(CARPETA_DATOS, clase)
    for nombre in os.listdir(carpeta):
        ruta = os.path.join(carpeta, nombre)

        # Revisión 1: cabecera JFIF
        with open(ruta, "rb") as f:
            es_jfif = b"JFIF" in f.peek(10)

        # Revisión 2: que se pueda decodificar con la misma función que se usa al entrenar
        se_puede_abrir = False
        if es_jfif:
            try:
                tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
                se_puede_abrir = True
            except Exception:
                pass

        if not (es_jfif and se_puede_abrir):
            os.remove(ruta)
            eliminadas[clase] += 1

print("Imágenes eliminadas:", eliminadas, "| total:", sum(eliminadas.values()))

## 3. Balanceo de clases

Después de limpiar no queda la misma cantidad de gatos que de perros. Para que el modelo no se incline
por la clase más numerosa, se toma de cada clase la cantidad de la clase más pequeña (*undersampling*).

In [ ]:
cat_files = sorted(glob.glob(os.path.join(CARPETA_DATOS, "Cat", "*.jpg")))
dog_files = sorted(glob.glob(os.path.join(CARPETA_DATOS, "Dog", "*.jpg")))

# Se mezclan para que la selección sea al azar (con semilla, para que siempre dé lo mismo)
random.seed(SEMILLA)
random.shuffle(cat_files)
random.shuffle(dog_files)

num_cats = len(cat_files)
num_dogs = len(dog_files)
num_samples = min(num_cats, num_dogs)
if MUESTRA_POR_CLASE:
    num_samples = min(num_samples, MUESTRA_POR_CLASE)

cats = cat_files[:num_samples]
dogs = dog_files[:num_samples]

print("Antes del balanceo  -> gatos:", num_cats, "| perros:", num_dogs)
print("Después del balanceo -> gatos:", len(cats), "| perros:", len(dogs))

## 4. Partición: train 70 %, validation 15 %, test 15 %

- **train**: con estas imágenes aprende el modelo.
- **validation**: se usan durante el entrenamiento para vigilar el sobreajuste y escoger la mejor época.
- **test**: no se tocan hasta el final; dan la medida real de qué tan bueno es el modelo.

La partición se hace **por clase**, así los tres conjuntos quedan balanceados. Se guardan en archivos CSV
con dos columnas: `filename` (ruta de la imagen) y `label` (0 = gato, 1 = perro).

In [ ]:
def partir(archivos):
    n = len(archivos)
    fin_train = int(n * 0.70)      # los índices de una lista deben ser enteros
    fin_val = int(n * 0.85)
    return archivos[:fin_train], archivos[fin_train:fin_val], archivos[fin_val:]


def crear_dataframe(gatos, perros):
    df = pd.DataFrame({
        "filename": gatos + perros,
        "label": [0] * len(gatos) + [1] * len(perros),     # 0 = gato, 1 = perro
    })
    return df.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)   # mezcla gatos y perros


train_cats, val_cats, test_cats = partir(cats)
train_dogs, val_dogs, test_dogs = partir(dogs)

df_train = crear_dataframe(train_cats, train_dogs)
df_val = crear_dataframe(val_cats, val_dogs)
df_test = crear_dataframe(test_cats, test_dogs)

df_train.to_csv("train.csv", index=False)
df_val.to_csv("val.csv", index=False)
df_test.to_csv("test.csv", index=False)

# Comprobación: tamaños, balance y que ninguna imagen esté en dos conjuntos a la vez
total = len(df_train) + len(df_val) + len(df_test)
resumen = pd.DataFrame({
    "imágenes": [len(df_train), len(df_val), len(df_test)],
    "gatos": [(df_train.label == 0).sum(), (df_val.label == 0).sum(), (df_test.label == 0).sum()],
    "perros": [(df_train.label == 1).sum(), (df_val.label == 1).sum(), (df_test.label == 1).sum()],
}, index=["train", "val", "test"])
resumen["% del total"] = (100 * resumen["imágenes"] / total).round(1)

assert not set(df_train.filename) & set(df_val.filename)
assert not set(df_train.filename) & set(df_test.filename)
assert not set(df_val.filename) & set(df_test.filename)
print("Partición lista: ninguna imagen se repite entre conjuntos.")
resumen

## 5. Cargar las imágenes desde los CSV (`tf.data`)

Cada fila del CSV se convierte en un par (imagen, etiqueta): se lee el archivo, se decodifica y se ajusta a 180x180.

Dos detalles importantes:

- **No se divide por 255 aquí.** El modelo ya tiene una capa `Rescaling(1/255)` adentro. Si se dividiera dos veces,
  el modelo vería imágenes casi negras. Por eso el servidor Flask también le envía los píxeles de 0 a 255.
- Solo se mezcla (`shuffle`) el conjunto de entrenamiento. Validación y prueba se dejan en orden para poder
  comparar cada predicción con su etiqueta real.

In [ ]:
df_train = pd.read_csv("train.csv")
df_val = pd.read_csv("val.csv")
df_test = pd.read_csv("test.csv")


def process_path(filename, label):
    img = tf.io.read_file(filename)                                        # archivo -> bytes
    img = tf.io.decode_image(img, channels=3, expand_animations=False)     # bytes -> matriz RGB
    img = tf.image.resize(img, IMAGE_SIZE)                                 # -> 180x180
    return img, label


# Aumento de datos: voltear y rotar un poco las imágenes de entrenamiento
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
]


def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images


def crear_dataset(df, entrenamiento=False):
    ds = tf.data.Dataset.from_tensor_slices((df["filename"].values, df["label"].values))
    if entrenamiento:
        # Se mezclan las rutas (texto) y no las imágenes ya cargadas, para no llenar la memoria RAM
        ds = ds.shuffle(buffer_size=len(df), seed=SEMILLA)
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if entrenamiento:
        ds = ds.map(lambda img, label: (data_augmentation(img), label), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = crear_dataset(df_train, entrenamiento=True)
val_ds = crear_dataset(df_val)
test_ds = crear_dataset(df_test)

print("Lotes -> train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))

In [ ]:
# Un vistazo a 9 imágenes de entrenamiento (ya con aumento de datos)
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title("perro (1)" if int(labels[i]) == 1 else "gato (0)")
        plt.axis("off")
plt.show()

## 6. Modelo desde cero

Es la versión pequeña de la red Xception del ejemplo de Keras. La primera capa (`Rescaling`) pasa los píxeles
de 0-255 a 0-1, y la última es una sola neurona sin activación: entrega un *logit*, y con la función sigmoide
se obtiene la probabilidad de que sea **perro**.

In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Set aside residual

    for size in [256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Project residual
        residual = layers.Conv2D(size, 1, strides=2, padding="same")(previous_block_activation)
        x = layers.add([x, residual])  # Add back residual
        previous_block_activation = x  # Set aside next residual

    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    units = 1 if num_classes == 2 else num_classes

    x = layers.Dropout(0.25)(x)
    # activation=None: la salida es un logit
    outputs = layers.Dense(units, activation=None)(x)
    return keras.Model(inputs, outputs)


model = make_model(input_shape=IMAGE_SIZE + (3,), num_classes=2)
print("Parámetros del modelo:", f"{model.count_params():,}")

## 7. Entrenamiento con *callbacks*

- `ModelCheckpoint`: guarda en `mejor_modelo.keras` solo la época con menor pérdida de validación.
- `EarlyStopping`: detiene el entrenamiento si la validación deja de mejorar durante 7 épocas seguidas.
- `CSVLogger`: guarda las métricas de cada época en `historial_entrenamiento.csv`.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("mejor_modelo.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.CSVLogger("historial_entrenamiento.csv"),
]

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    validation_data=val_ds,
)

In [ ]:
# Curvas de entrenamiento: si 'val' se separa mucho de 'train', hay sobreajuste
h = history.history
mejor_epoca = int(np.argmin(h["val_loss"])) + 1

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, metrica, titulo in ((ejes[0], "loss", "Pérdida (loss)"), (ejes[1], "acc", "Exactitud (accuracy)")):
    epocas = range(1, len(h[metrica]) + 1)
    eje.plot(epocas, h[metrica], label="train")
    eje.plot(epocas, h["val_" + metrica], label="val")
    eje.axvline(mejor_epoca, color="gray", linestyle="--", label="mejor época")
    eje.set_title(titulo)
    eje.set_xlabel("época")
    eje.legend()
plt.show()

print(f"Mejor época: {mejor_epoca} | val_loss = {h['val_loss'][mejor_epoca - 1]:.4f} | val_acc = {h['val_acc'][mejor_epoca - 1]:.4f}")

## 8. Evaluación con el conjunto de prueba

Se carga la mejor época y se evalúa con las imágenes de **test**, que el modelo nunca vio.
Se toma **perro (1) como la clase positiva**:

| | Predijo gato | Predijo perro |
|---|---|---|
| **Era gato** | TN (verdadero negativo) | FP (falso positivo) |
| **Era perro** | FN (falso negativo) | TP (verdadero positivo) |

- **Accuracy (exactitud)** = (TP + TN) / total → de todas las imágenes, cuántas acertó.
- **Precision** = TP / (TP + FP) → cuando dice "perro", cuántas veces es verdad.
- **Recall (sensibilidad)** = TP / (TP + FN) → de todos los perros, cuántos encontró.
- **Specificity (especificidad)** = TN / (TN + FP) → de todos los gatos, cuántos reconoció como gatos.
- **F1** = media armónica entre precision y recall.

In [ ]:
model = keras.models.load_model("mejor_modelo.keras")

test_loss, test_acc = model.evaluate(test_ds)

# Predicciones: logit -> probabilidad de perro (sigmoide) -> clase con umbral 0.5
logits = model.predict(test_ds).ravel()
prob_perro = 1 / (1 + np.exp(-logits))
y_pred = (prob_perro >= 0.5).astype(int)
y_true = df_test["label"].values

TP = int(((y_true == 1) & (y_pred == 1)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) else 0.0
recall = TP / (TP + FN) if (TP + FN) else 0.0
specificity = TN / (TN + FP) if (TN + FP) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

metricas = pd.DataFrame({
    "métrica": ["accuracy", "precision", "recall", "specificity", "f1", "loss"],
    "valor": [accuracy, precision, recall, specificity, f1, test_loss],
}).round(4)
metricas.to_csv("metricas_test.csv", index=False)
metricas

In [ ]:
# Matriz de confusión
matriz = np.array([[TN, FP], [FN, TP]])

plt.figure(figsize=(4.5, 4))
plt.imshow(matriz, cmap="Blues")
plt.xticks([0, 1], ["gato", "perro"])
plt.yticks([0, 1], ["gato", "perro"])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión (test)")
for i in range(2):
    for j in range(2):
        color = "white" if matriz[i, j] > matriz.max() / 2 else "black"
        plt.text(j, i, matriz[i, j], ha="center", va="center", color=color, fontsize=14)
plt.show()

## 9. Guardar el modelo para el servidor

Se guarda como `modelo.keras`. Ese archivo se copia a la carpeta `proyecto_vision` (reemplazando el anterior)
y se reinicia `server.py`. Antes se prueba una imagen igual que lo hace el servidor.

In [ ]:
model.save("modelo.keras")

# Prueba de inferencia con una imagen de test, igual que en el servidor Flask
ruta = df_test["filename"][0]
img = keras.utils.load_img(ruta, target_size=IMAGE_SIZE)
img_array = np.expand_dims(keras.utils.img_to_array(img), 0)       # forma (1, 180, 180, 3), valores 0-255
score = float(keras.ops.sigmoid(model.predict(img_array, verbose=0)[0][0]))
print(f"{ruta}: {100 * (1 - score):.2f}% gato y {100 * score:.2f}% perro (etiqueta real: {df_test['label'][0]})")

try:
    from google.colab import files       # en Colab se descarga al computador
    files.download("modelo.keras")
except ImportError:
    print("Modelo guardado en:", os.path.abspath("modelo.keras"))

## 10. (Opcional) Comparación con un modelo pre-entrenado

En clase también se vieron los modelos pre-entrenados de `keras.applications` (DenseNet121, ResNet50),
que ya aprendieron con ImageNet. Con **transferencia de aprendizaje** se reutiliza esa red congelada y solo
se entrena una capa final nueva para nuestras dos clases.

Nota: `include_top=True` con `weights="imagenet"` solo funciona con las 1000 clases de ImageNet; para
2 clases se usa `include_top=False` y se agrega la capa de salida propia.

Cambia `COMPARAR_PREENTRENADO` a `True` para ejecutarlo (unos 10 minutos más con GPU).

In [ ]:
COMPARAR_PREENTRENADO = False
EPOCHS_PREENTRENADO = 3

if COMPARAR_PREENTRENADO:
    base = keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    base.trainable = False                       # se congela lo que ya aprendió con ImageNet

    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    x = layers.Rescaling(1.0 / 255)(inputs)      # mismo preprocesamiento con el que se entrenó DenseNet
    x = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation=None)(x)
    model2 = keras.Model(inputs, outputs, name="densenet121_transfer")

    model2.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[keras.metrics.BinaryAccuracy(name="acc")],
    )
    model2.fit(train_ds, epochs=EPOCHS_PREENTRENADO, validation_data=val_ds)

    evaluation1 = model.evaluate(test_ds, verbose=0)
    evaluation2 = model2.evaluate(test_ds, verbose=0)
    model2.save("modelo_densenet121.keras")

    comparacion = pd.DataFrame({
        "modelo": ["Desde cero (Xception pequeña)", "DenseNet121 pre-entrenado"],
        "test_loss": [evaluation1[0], evaluation2[0]],
        "test_acc": [evaluation1[1], evaluation2[1]],
        "parámetros": [model.count_params(), model2.count_params()],
    }).round(4)
    display(comparacion)
else:
    print("Comparación omitida. Cambia COMPARAR_PREENTRENADO a True para ejecutarla.")